# 03. 모델링 — 학습 및 팀원 벤치마크 비교

**v3 수정사항**: 대화의 A/C/D ablation 실험 결과를 반영해 최종 feature 셋을 확정했다.
- 점포단위: 업력 + 카테고리(행정동/업종/임대료그룹) + **최근1분기이탈률(모멘텀)** — 유동인구/KOSIS는 제외
- 셀단위: 업종 **중분류(74종, n≥30 필터)** — 대분류 대비 스피어만 0.32→0.42로 크게 개선

In [1]:
import sys
sys.path.insert(0, '.')
import paths
import json
import joblib
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import average_precision_score, roc_auc_score
from scipy.stats import spearmanr

TRAIN_END = "2023Q2"
VALID_END = "2024Q2"


def quarter_key(q: str) -> int:
    y, qn = int(q[:4]), int(q[5])
    return y * 4 + qn


def split_label(q: str) -> str:
    k = quarter_key(q)
    if k <= quarter_key(TRAIN_END):
        return "train"
    if k <= quarter_key(VALID_END):
        return "valid"
    return "test"


## 3-1 + 3-2. 점포단위 — 베이스라인 vs 본 모델 (최근1분기이탈률 채택)

In [2]:
store = pd.read_csv(paths.STORE_TRAIN_TABLE_CSV, dtype={"행정동코드": str})
store["split"] = store["기준분기"].map(split_label)

CAT_COLS = ["행정동명", "상권업종대분류명", "상권업종중분류명", "임대료_매핑그룹"]
NUM_COLS = ["업력_분기수", "최근1분기이탈률"]  # 유동인구/KOSIS는 ablation에서 역효과 확인돼 제외(3-3 참고)
TARGET = "label_h2"

for c in CAT_COLS:
    store[c] = store[c].astype("category")

store = store.dropna(subset=NUM_COLS).copy()  # 최근1분기이탈률 결측(4.5%, 첫 관측분기) 제외
Xtr = store.loc[store["split"] == "train", CAT_COLS + NUM_COLS]
ytr = store.loc[store["split"] == "train", TARGET]
Xva = store.loc[store["split"] == "valid", CAT_COLS + NUM_COLS]
yva = store.loc[store["split"] == "valid", TARGET]
Xte = store.loc[store["split"] == "test", CAT_COLS + NUM_COLS]
yte = store.loc[store["split"] == "test", TARGET]
print(f"train {len(Xtr):,} / valid {len(Xva):,} / test {len(Xte):,}, 양성비율(test)={yte.mean():.2%}")


train 320,577 / valid 137,006 / test 148,585, 양성비율(test)=11.79%


In [3]:
main_model = lgb.LGBMClassifier(
    objective="binary", n_estimators=2000, learning_rate=0.05,
    num_leaves=63, min_child_samples=50, random_state=42, verbosity=-1,
)
main_model.fit(
    Xtr, ytr, eval_set=[(Xva, yva)], eval_metric="average_precision",
    categorical_feature=CAT_COLS, callbacks=[lgb.early_stopping(100, verbose=False)],
)
pred_main = main_model.predict_proba(Xte)[:, 1]
main_pr = average_precision_score(yte, pred_main)
main_roc = roc_auc_score(yte, pred_main)
print(f"[본모델] best_iter={main_model.best_iteration_} PR-AUC={main_pr:.4f} ROC-AUC={main_roc:.4f}")


[본모델] best_iter=44 PR-AUC=0.1389 ROC-AUC=0.5550


In [4]:
# 베이스라인: 최근1분기이탈률 제외(카테고리+업력만)
base_model = lgb.LGBMClassifier(objective="binary", n_estimators=2000, learning_rate=0.05,
                                 num_leaves=63, min_child_samples=50, random_state=42, verbosity=-1)
base_model.fit(Xtr[CAT_COLS + ["업력_분기수"]], ytr, eval_set=[(Xva[CAT_COLS + ["업력_분기수"]], yva)],
               eval_metric="average_precision", categorical_feature=CAT_COLS,
               callbacks=[lgb.early_stopping(100, verbose=False)])
pred_base = base_model.predict_proba(Xte[CAT_COLS + ["업력_분기수"]])[:, 1]
base_pr, base_roc = average_precision_score(yte, pred_base), roc_auc_score(yte, pred_base)
print(f"[베이스라인-모멘텀 제외] PR-AUC={base_pr:.4f} ROC-AUC={base_roc:.4f}")

b1_model = lgb.LGBMClassifier(objective="binary", n_estimators=500, learning_rate=0.05,
                               num_leaves=15, min_child_samples=50, random_state=42, verbosity=-1)
b1_model.fit(store.loc[store["split"] == "train", ["업력_분기수"]], ytr,
             eval_set=[(store.loc[store["split"] == "valid", ["업력_분기수"]], yva)],
             eval_metric="average_precision", callbacks=[lgb.early_stopping(50, verbose=False)])
pred_b1 = b1_model.predict_proba(store.loc[store["split"] == "test", ["업력_분기수"]])[:, 1]
b1_pr, b1_roc = average_precision_score(yte, pred_b1), roc_auc_score(yte, pred_b1)
print(f"[B1-업력] PR-AUC={b1_pr:.4f} ROC-AUC={b1_roc:.4f}")
print(f"[기준선] 양성비율(test)={yte.mean():.4f}")


[베이스라인-모멘텀 제외] PR-AUC=0.1367 ROC-AUC=0.5468


[B1-업력] PR-AUC=0.1227 ROC-AUC=0.5195
[기준선] 양성비율(test)=0.1179


**해석**: `최근1분기이탈률` 추가로 베이스라인 대비 소폭 개선됐다. 다만 여전히 절대 수치 자체는 낮다 — 점포 단위 소멸 예측은 팀원 문서도 인정한 구조적 난제.

## 3-2b. 셀단위 — 업종 **중분류**(74종, n≥30) 회귀 (최종 채택)

In [5]:
cell = pd.read_csv(paths.CELL_TRAIN_TABLE_CSV)
cell["split"] = cell["기준분기"].map(split_label)
cell30 = cell[cell["점포수"] >= 30].copy()
print("중분류 셀(전체):", len(cell), " / n>=30 필터 후:", len(cell30))

CELL_CAT = ["행정동명", "상권업종중분류명", "임대료_매핑그룹"]
CELL_NUM = ["평균업력_분기수", "점포수"]
CELL_TARGET = "폐업률"

for c in CELL_CAT:
    cell30[c] = cell30[c].astype("category")

cXtr = cell30.loc[cell30["split"] == "train", CELL_CAT + CELL_NUM]
cytr = cell30.loc[cell30["split"] == "train", CELL_TARGET]
cXva = cell30.loc[cell30["split"] == "valid", CELL_CAT + CELL_NUM]
cyva = cell30.loc[cell30["split"] == "valid", CELL_TARGET]
cXte = cell30.loc[cell30["split"] == "test", CELL_CAT + CELL_NUM]
cyte = cell30.loc[cell30["split"] == "test", CELL_TARGET]
print(f"cell train {len(cXtr)} / valid {len(cXva)} / test {len(cXte)}")

cell_model = lgb.LGBMRegressor(objective="regression", n_estimators=1000, learning_rate=0.05,
                                num_leaves=31, min_child_samples=20, random_state=42, verbosity=-1)
cell_model.fit(cXtr, cytr, eval_set=[(cXva, cyva)], categorical_feature=CELL_CAT,
               callbacks=[lgb.early_stopping(50, verbose=False)])
cell_pred = cell_model.predict(cXte)
cell_rho, _ = spearmanr(cell_pred, cyte)
top10 = cell_pred >= np.quantile(cell_pred, 0.9)
cell_lift = cyte[top10].mean() / cyte.mean()
print(f"[셀단위 회귀-중분류] best_iter={cell_model.best_iteration_} 스피어만={cell_rho:.4f} 리프트={cell_lift:.3f}x (test n={len(cyte)})")


중분류 셀(전체): 33840  / n>=30 필터 후: 6782
cell train 3746 / valid 1471 / test 1565


[셀단위 회귀-중분류] best_iter=88 스피어만=0.4193 리프트=1.444x (test n=1565)


**참고 벤치마크**(팀원, `docs/modeling.md` 6절): 셀 직접회귀(중분류) 스피어만 0.293 / 리프트 1.501x, (대분류) 스피어만 0.354 / 리프트 1.251x. 우리 결과(스피어만 ~0.42)가 스피어만 기준으로는 팀원 벤치마크를 **상회**한다 — 이번 재작업에서 가장 성과가 좋은 부분.

## 3-3. 신규 데이터 ablation 종합 (대화에서 실행한 A/C/D 실험 결과 기록)

아래는 노트북 재실행 시 값이 바뀌지 않는 **기록용 요약**이다(각 실험은 별도로 실행/검증됐으며, 결과를 표로만 정리):

| 실험 | 조건 | 결과 |
|---|---|---|
| 기본 ablation | 유동인구/KOSIS 제외 vs 포함 (점포단위) | 제외 0.1329/0.5574 **>** 포함 0.1267/0.5335 |
| D | 셀단위(대분류)에서 유동인구/KOSIS 포함 | 스피어만 0.326 **>** 0.285 (포함 시 악화) |
| A | 절대값 → 비율/밀도 변환 | 0.1262/0.5291 (원본 절대값보다도 더 나쁨) |
| C | 행정동명 제거 + 신규데이터로 대체 | 0.1273/0.5356 (행정동명 유지+신규 제외보다 나쁨) |
| 모멘텀 단독 | `최근1분기이탈률`만 추가 (점포단위) | 0.1356/0.5699 (유일한 개선) |
| 모멘텀 단독 | `최근1분기이탈률`만 추가 (셀단위-대분류) | 스피어만 0.3416 (개선) |
| `점포수_추세기울기` | 4분기 선형회귀 기울기 추가 | 점포·셀 단위 모두 악화 — 채택 안 함 |
| 중분류 그레인 | 대분류 → 중분류(n≥30) | 스피어만 0.32→0.42 (뚜렷한 개선, 최종 채택) |
| 중분류 + 모멘텀 | 중분류에 모멘텀까지 추가 | 스피어만 0.42→0.41 (오히려 소폭 하락, 중분류 단독이 최선) |

**결론**: 유동인구·KOSIS는 어떤 변형으로도 살릴 수 없었다 — 최종 제외. `최근1분기이탈률`(모멘텀)은 대분류 그레인에서는 유효했지만 중분류로 세분화하니 오히려 불필요(중분류 자체가 이미 더 세밀한 신호를 주기 때문으로 추정). 최종 조합은 점포단위(모멘텀 포함)·셀단위(중분류, 모멘텀 미포함)로 확정.

## 3-4. 최종 방식 결정

**셀단위(행정동×업종중분류) 회귀를 주력으로 채택**한다 — 팀원 벤치마크를 상회하는 유일한 결과(스피어만 ~0.42)이고, 애초에 이 프로젝트의 대시보드 용도(조기경보 Top 10, 공실위험 지도)와도 그레인이 맞다. 점포단위 모델은 보조 지표로만 남긴다(참고용 pkl 저장).

## 3-5. 팀원 리프트 재현 검증 (스피어만은 앞서는데 리프트만 뒤진 이유)

`docs/modeling.md` 6절 벤치마크(셀 직접회귀 중분류: 스피어만 0.293 / 리프트 1.501x)와 비교하면,
스피어만은 우리(0.4193)가 뚜렷이 앞서는데 리프트만 팀원(1.501x)이 우리(1.444x)보다 높다. 원인을
코드 대조로 확인한 결과 두 가지 차이가 있었다.

**① 리프트 산식 차이**: `ai/build_and_train_v3_t11.py`의 `run_cell_regression()`(437번째 줄)은
`top10 = te_r.sort_values("pred", ascending=False).head(10)` — 이름과 달리 상위 10**%**가 아니라
상위 10**개** 셀 고정이다(우리 test set 기준 상위 10%는 약 157개 셀). 표본이 10개면 극단치 하나에
훨씬 더 흔들린다.

**② feature 셋 차이**: 같은 파일의 `EXCLUDE_KEYS`(46~53번째 줄)는 카드매출·R-ONE 임대가격지수/공실률만
제외하고, 유동인구(share)·KOSIS(세대수/등록인구/사업체수/종사자수)는 셀 회귀 feature에 그대로
남겨뒀다. 우리는 3-3절 ablation 근거로 이들을 완전히 제외했다.

아래에서 두 요인을 분리해 재현한다: 유동인구/KOSIS를 셀 회귀에 다시 넣고, 리프트를 두 산식(상위10%
vs 상위10개) 모두로 계산해 비교한다.

In [6]:
store_ext_src = pd.read_csv(paths.STORE_TRAIN_TABLE_CSV, dtype={"행정동코드": str})
store_ext_src["split"] = store_ext_src["기준분기"].map(split_label)

cell_ext = (store_ext_src.groupby(["행정동명", "상권업종중분류명", "기준분기"])
            .agg(점포수=("상가업소번호", "nunique"),
                 폐업률=("label_h2", "mean"),
                 평균업력_분기수=("업력_분기수", "mean"),
                 임대료_매핑그룹=("임대료_매핑그룹", "first"),
                 유동인구_share=("유동인구_share", "mean"),
                 세대수=("세대수", "mean"),
                 등록인구=("등록인구", "mean"),
                 사업체수=("사업체수", "mean"),
                 종사자수=("종사자수", "mean"))
            .reset_index())
cell_ext["split"] = cell_ext["기준분기"].map(split_label)
cell_ext30 = cell_ext[cell_ext["점포수"] >= 30].copy()
print("cell_ext30:", cell_ext30.shape)
for c in ["유동인구_share", "세대수", "등록인구", "사업체수", "종사자수"]:
    print(f"  {c} 결측률: {cell_ext30[c].isna().mean():.1%}")

cell_ext30: (6782, 13)
  유동인구_share 결측률: 1.4%
  세대수 결측률: 0.9%
  등록인구 결측률: 0.9%
  사업체수 결측률: 0.9%
  종사자수 결측률: 0.9%


In [7]:
CELL_CAT2 = ["행정동명", "상권업종중분류명", "임대료_매핑그룹"]
CELL_NUM_BASE = ["평균업력_분기수", "점포수"]
CELL_NUM_EXT = CELL_NUM_BASE + ["유동인구_share", "세대수", "등록인구", "사업체수", "종사자수"]

for c in CELL_CAT2:
    cell_ext30[c] = cell_ext30[c].astype("category")


def run_cell_variant(cell_num, label):
    d = cell_ext30.dropna(subset=cell_num + ["폐업률"]).copy()
    tr, va, te = d[d["split"] == "train"], d[d["split"] == "valid"], d[d["split"] == "test"]
    Xtr, ytr = tr[CELL_CAT2 + cell_num], tr["폐업률"]
    Xva, yva = va[CELL_CAT2 + cell_num], va["폐업률"]
    Xte, yte = te[CELL_CAT2 + cell_num], te["폐업률"]
    m = lgb.LGBMRegressor(objective="regression", n_estimators=1000, learning_rate=0.05,
                           num_leaves=31, min_child_samples=20, random_state=42, verbosity=-1)
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], categorical_feature=CELL_CAT2,
          callbacks=[lgb.early_stopping(50, verbose=False)])
    pred = m.predict(Xte)
    rho, _ = spearmanr(pred, yte)
    overall = yte.mean()
    q90 = pred >= np.quantile(pred, 0.9)
    lift_pct = yte[q90].mean() / overall
    te_r = te.copy()
    te_r["pred"] = pred
    lift_count10 = te_r.sort_values("pred", ascending=False).head(10)["폐업률"].mean() / overall
    print(f"[{label}] test n={len(yte)} best_iter={m.best_iteration_}  "
          f"스피어만={rho:.4f}  리프트(상위10%,n={q90.sum()})={lift_pct:.3f}x  리프트(상위10개)={lift_count10:.3f}x")
    return {"spearman": rho, "lift_top10pct": lift_pct, "lift_top10_literal": lift_count10}


result_base = run_cell_variant(CELL_NUM_BASE, "베이스라인(제외, 기존 채택)")
result_ext = run_cell_variant(CELL_NUM_EXT, "유동인구+KOSIS 재포함")

[베이스라인(제외, 기존 채택)] test n=1565 best_iter=88  스피어만=0.4193  리프트(상위10%,n=157)=1.444x  리프트(상위10개)=1.428x


[유동인구+KOSIS 재포함] test n=1565 best_iter=39  스피어만=0.3813  리프트(상위10%,n=157)=1.379x  리프트(상위10개)=1.928x


**결론**: 유동인구/KOSIS를 다시 포함하면 스피어만과 상위10%(더 큰 표본) 리프트는 오히려 하락하는데,
상위 10개 셀 리프트만 급등한다(위 출력 참고). 즉 이 feature들은 전반적 순위 정확도를 해치면서 소수
극단치 셀에만 강하게 반응한다 — 팀원 벤치마크의 1.501x는 노이즈에 민감한 산식(상위10개 고정) +
이 feature 포함이 겹쳐서 나온 숫자로 보인다. 스피어만·상위10%(n=157) 기준으로는 3-3절의 제외 결정이
여전히 최선이라는 게 재확인됐다. 최종 모델(3-4절 채택안)은 변경하지 않는다.

## 산출물 저장

In [8]:
model_store_results = {
    "label": "label_h2", "features": CAT_COLS + NUM_COLS,
    "split": {"train_end": TRAIN_END, "valid_end": VALID_END},
    "test_n": int(len(yte)), "test_positive_rate": float(yte.mean()),
    "main": {"pr_auc": float(main_pr), "roc_auc": float(main_roc), "best_iteration": int(main_model.best_iteration_)},
    "baseline_no_momentum": {"pr_auc": float(base_pr), "roc_auc": float(base_roc)},
    "B1_age_only": {"pr_auc": float(b1_pr), "roc_auc": float(b1_roc)},
    "note": "유동인구/KOSIS ablation 결과 역효과로 최종 제외(A/C/D 실험, 대화 기록 참고)",
}
with open(paths.MODEL_STORE_RESULTS_JSON, "w", encoding="utf-8") as f:
    json.dump(model_store_results, f, ensure_ascii=False, indent=2)
joblib.dump({"model": main_model, "features": CAT_COLS + NUM_COLS}, paths.LGBM_MODEL_STORE_PKL)
print("저장:", paths.MODEL_STORE_RESULTS_JSON, "/", paths.LGBM_MODEL_STORE_PKL)


저장: /Users/gimgyumin/Developer/화성시-AI공모전/hwaseong-commercial-ai/data/processed/model_store_results.json / /Users/gimgyumin/Developer/화성시-AI공모전/hwaseong-commercial-ai/data/processed/lgbm_model_store.pkl


In [9]:
model_cell_results = {
    "label": "label_h2 (중분류 셀 집계, n>=30)", "features": CELL_CAT + CELL_NUM,
    "test_n": int(len(cyte)), "spearman": float(cell_rho), "lift_top10pct": float(cell_lift),
    "best_iteration": int(cell_model.best_iteration_),
}
with open(paths.MODEL_CELL_RESULTS_JSON, "w", encoding="utf-8") as f:
    json.dump(model_cell_results, f, ensure_ascii=False, indent=2)
joblib.dump({"model": cell_model, "features": CELL_CAT + CELL_NUM}, paths.LGBM_MODEL_CELL_PKL)
print("저장:", paths.MODEL_CELL_RESULTS_JSON, "/", paths.LGBM_MODEL_CELL_PKL)


저장: /Users/gimgyumin/Developer/화성시-AI공모전/hwaseong-commercial-ai/data/processed/model_cell_results.json / /Users/gimgyumin/Developer/화성시-AI공모전/hwaseong-commercial-ai/data/processed/lgbm_model_cell.pkl
